In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://develop.rdm.example.com/'
rdm_admin_url = 'https://admin.develop.rdm.example.com/'
idp_name_1 = "GakuNin RDM IdP"
idp_username_1 = 'user_test_admin'
idp_password_1 = 'password_test_admin'
admin_idp_username_1 = 'user_test_admin'
admin_idp_password_1 = 'password_test_admin'
rdm_project_name = None
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
delete_project = True
run_id = datetime.now().strftime('%Y%m%d-%H%M%S')
default_result_path = f'result/result-{run_id}'
close_on_fail = False
transition_timeout = 60000
skip_failed_test = False
exclude_notebooks = []

In [2]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

(7, 16)

In [3]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

'/tmp/tmpzdr6hzm3'

# GakuNinRDM 総合テスト [グループ管理連携機能]

- サブシステム名: グループ管理連携機能
- ページ/アドオン: グループ
- 機能分類: グループ管理連携機能
- シナリオ名: *
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM)

In [ ]:
from datetime import datetime
import os
import papermill as pm
import traceback
from scripts.papermillHelpers import gen_run_notebook

def make_result_dir(base_path):
    result_dir = os.path.join(base_path, 'notebooks')
    os.makedirs(result_dir, exist_ok=True)
    return result_dir

result_dir = make_result_dir(default_result_path)

run_notebook = gen_run_notebook(
    result_dir,
    transition_timeout,
    dict(
        rdm_url=rdm_url,
        idp_name_1=idp_name_1,
        idp_username_1=idp_username_1,
        idp_password_1=idp_password_1,
    ),
    skip_failed_test,
)

result_notebooks = []
result_dir

'result/result-20260330-043426'

## 「アドオン利用制御」テストの実施

テスト「テスト手順-グループ-アドオン利用制御」を実施する。

In [ ]:
result_notebooks.append(run_notebook('テスト手順-グループ-アドオン利用制御.ipynb',
    dict(
        rdm_admin_url=rdm_admin_url,
        idp_username_1=admin_idp_username_1,
        idp_password_1=admin_idp_password_1,
    )
))
result_notebooks[-1]

Executing:   0%|          | 0/16 [00:00<?, ?cell/s]

'result/result-20260330-043426/テスト手順-グループ-アドオン利用制御.ipynb'

## 「プロジェクトに対するグループ機能①」テストの実施

テスト「テスト手順-プロジェクトに対するグループ機能」を実施する。

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-グループ-プロジェクトに対するグループ機能①.ipynb',
))
result_notebooks[-1]

Executing:   0%|          | 0/122 [00:00<?, ?cell/s]

'result/result-20260330-043426/テスト手順-グループ-プロジェクトに対するグループ機能①.ipynb'

## 「グループ権限によるプロジェクトに対する操作」テストの実施

テスト「テスト手順-グループ権限によるプロジェクトに対する操作」を実施する。

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-グループ-グループ権限によるプロジェクトに対する操作.ipynb',
))
result_notebooks[-1]

Executing:   0%|          | 0/98 [00:00<?, ?cell/s]

'result/result-20260330-043426/テスト手順-グループ-グループ権限によるプロジェクトに対する操作.ipynb'

## 「グループ権限によるコンポーネントに対する操作」テストの実施

テスト「テスト手順-グループ権限によるコンポーネントに対する操作」を実施する。

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-グループ-グループ権限によるコンポーネントに対する操作.ipynb',
))
result_notebooks[-1]

Executing:   0%|          | 0/141 [00:00<?, ?cell/s]

## 「プロジェクトに対するグループ機能②」テストの実施

テスト「テスト手順-プロジェクトに対するグループ機能」を実施する。

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-グループ-プロジェクトに対するグループ機能②.ipynb',
))
result_notebooks[-1]

'result/result-20260330-043426/テスト手順-グループ-プロジェクトに対するグループ機能②.ipynb'

# 結果の収集

In [10]:
import nbformat
import re

def has_header1(cell):
    if cell['cell_type'] != 'markdown':
        return False
    line = cell['source'].split('\n')[0]
    m = re.match(r'#\s+(.+)', line)
    if not m:
        return False
    return True

def is_test_set(cell):
    if cell['cell_type'] != 'markdown':
        return False
    line = cell['source'].split('\n')[0]
    m = re.match(r'#\s+(.+)', line)
    if not m:
        return False
    if m.group(1) == '報告書出力':
        return False
    return True

def parse_cells(notebook_file):
    notebook = nbformat.read(
        notebook_file,
        as_version=nbformat.NO_CONVERT
    )
    cells = notebook['cells']
    test_sets = []
    
    for i, cell in enumerate(cells):
        if not has_header1(cell):
            continue
        if is_test_set(cell):
            test_sets.append((i, cell))
        else:
            test_sets.append((i, None))
            break
    if len(test_sets) == 0 or test_sets[-1][-1] is not None:
        test_sets.append((len(cells), None))
    return (cells, test_sets)

all_test_sets = []

for notebook_file in result_notebooks:
    print(notebook_file)
    all_test_sets.append((notebook_file, parse_cells(notebook_file)))
    notebook_dir, _ = os.path.split(notebook_file)
    notebooks_dir = os.path.join(notebook_dir, os.path.splitext(os.path.split(notebook_file)[-1])[0], 'notebooks')
    if not os.path.isdir(notebooks_dir):
        continue
    for filename in os.listdir(notebooks_dir):
        if os.path.splitext(filename)[-1] != '.ipynb':
            continue
        child_notebook_file = os.path.join(notebooks_dir, filename)
        print(child_notebook_file)
        all_test_sets.append((child_notebook_file, parse_cells(child_notebook_file)))
len(all_test_sets)

5

In [11]:
id_prefix = ''
ticket_number = ''
author = 'Testrates'

In [12]:
import openpyxl
from openpyxl.styles import Alignment
from openpyxl.styles import PatternFill
from openpyxl.drawing.image import Image
from base64 import b64decode
import shutil

fill = PatternFill(start_color="AED6F1", fill_type="solid")

def save_image(cellindex, base64data):
    filename = os.path.join(work_dir, f'screenshot-{cellindex}.png')
    with open(filename, 'wb') as f:
        f.write(b64decode(base64data))
    return filename

def get_images_from_cell(cellindex, cell):
    if 'outputs' not in cell:
        return None
    images = [out['data']['image/png'] for out in cell['outputs'] if 'data' in out and 'image/png' in out['data']]
    return [save_image(cellindex, image) for image in images]

def has_header2(cell):
    if cell['cell_type'] != 'markdown':
        return False
    line = cell['source'].split('\n')[0]
    m = re.match(r'##\s+(.+)', line)
    if not m:
        return False
    return True

wb = openpyxl.Workbook()
summary_sheet = wb.worksheets[0]
summary_sheet.title = 'サマリ'

summary_sheet.column_dimensions['A'].width = summary_sheet.column_dimensions['A'].width * 1.25
for colname, text in zip('ABCDEFGHIJKLMNO', ['ID', 'シート', 'サブシステム', 'ページ/アドオン', '機能分類', 'シナリオ名', '概要', 'リンク', 'テスト結果', '関連チケット', '担当', '実施日', 'コメント', '修正確認', '確認日']):
    summary_sheet.column_dimensions[colname].width = summary_sheet.column_dimensions['A'].width
    summary_sheet[f'{colname}1'] = text
    summary_sheet[f'{colname}1'].fill = fill

index = 0
for notebook_file, (cells, test_sets) in all_test_sets:
    print(notebook_file)
    sheetname = '_'.join(os.path.splitext(os.path.split(notebook_file)[-1])[0].split('-')[1:][::-1][:2])
    for i, ((start, header), (end, _)) in enumerate(zip(test_sets, test_sets[1:])):
        index += 1
        test_id = f'{id_prefix}{index:03d}{sheetname}'
        os.makedirs(os.path.join(result_dir, 'screenshots', test_id), exist_ok=True)
        video_path, _ = os.path.splitext(notebook_file)
        if os.path.isdir(video_path):
            for video_file in os.listdir(video_path):
                if os.path.splitext(video_file)[-1].lower() != '.webm':
                    continue
                shutil.copyfile(os.path.join(video_path, video_file), os.path.join(result_dir, 'screenshots', test_id, video_file))
        line = header['source'].split('\n')[0]
        m = re.match(r'#\s+(.+)', line)
        title = m.group(1)
        attrs = {}
        for line in header['source'].split('\n'):
            m = re.match(r'-\s+([^:]+):\s*(.+)', line)
            if not m:
                continue
            attrs[m.group(1)] = m.group(2)
        sheet = wb.create_sheet(test_id)
        print('Sheet', test_id, start, end, title, attrs)
        
        for colname in 'ABCDEFGHIJ':
            sheet[f'{colname}1'].fill = fill
            sheet[f'{colname}4'].fill = fill
        sheet['A6'].fill = fill
    
        sheet['A1'] = 'ID'
        sheet['A2'] = test_id
        sheet['B1'] = 'サブシステム名'
        sheet['B2'] = attrs['サブシステム名']
        sheet['C1'] = '分類'
        sheet['C2'] = attrs['機能分類']
        sheet['D2'] = attrs['ページ/アドオン']
        sheet['H1'] = '作成者'
        sheet['H2'] = author
        sheet['I1'] = '作成日'
        sheet['I2'] = datetime.now().strftime('%Y-%m-%d')
        sheet['J1'] = '修正日'
        sheet['J2'] = ''
    
        sheet['A4'] = '概要'
        sheet['A5'] = attrs['概要'] if '概要' in attrs else title
        sheet['A5'].alignment = Alignment(wrap_text=True)
        sheet.merge_cells('A4:B4')
        sheet.merge_cells('A5:B5')
        sheet['C4'] = '用意するテストデータ'
        sheet['C5'] = attrs['用意するテストデータ']
        sheet['C5'].alignment = Alignment(wrap_text=True)
        sheet['D4'] = 'テスト結果'
        sheet['D5'] = '成功'
        sheet['E4'] = '関連チケットURL'
        sheet['E5'] = f'GRDM-{ticket_number}'
        sheet['E5'].hyperlink = f'https://redmine.devops.rcos.nii.ac.jp/issues/{ticket_number}'
        sheet['F4'] = '担当'
        sheet['F5'] = author
        sheet['G4'] = '実施日'
        sheet['G5'] = datetime.now().strftime('%Y-%m-%d')
        sheet['H4'] = 'コメント'
        sheet['H5'] = ''
        sheet['I4'] = '修正確認'
        sheet['I5'] = ''
        sheet['J4'] = '確認日'
        sheet['J5'] = ''
        for cell in sheet['A5:J5'][0]:
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    
        sheet['A6'] = '確認環境'
        sheet['B6'] = 'Ubuntu'
        sheet['C6'] = 'Chrome(Playwright)'
        sheet['D6'] = 'ja-JP'
    
        sheet.column_dimensions['B'].width = sheet.column_dimensions['A'].width * 4
        sheet.column_dimensions['C'].width = sheet.column_dimensions['A'].width * 5
        sheet.column_dimensions['E'].width = sheet.column_dimensions['A'].width * 2
        sheet.column_dimensions['G'].width = sheet.column_dimensions['A'].width * 2
        sheet.column_dimensions['H'].width = sheet.column_dimensions['A'].width * 2
        sheet.column_dimensions['I'].width = sheet.column_dimensions['A'].width * 2
        sheet.column_dimensions['J'].width = sheet.column_dimensions['A'].width * 2
        sheet.row_dimensions[5].height = sheet.column_dimensions['A'].width * 3
    
        startrow = 8
        sheet[f'A{startrow}'] = 'No.'
        sheet[f'B{startrow}'] = 'テスト手順'
        sheet[f'C{startrow}'] = '確認内容'
        sheet[f'D{startrow}'] = '実施'
        sheet[f'E{startrow}'] = 'コメント'
        sheet[f'F{startrow}'] = '実施者'
        sheet[f'G{startrow}'] = '実施日'
        sheet[f'H{startrow}'] = 'スクリーンショット'
        for colname in 'ABCDEFGHIJ':
            sheet[f'{colname}{startrow}'].fill = fill
    
        itemheight = sheet.column_dimensions['A'].width * 12 #* 6
        itemindex = 1
        last_images = None
        row = startrow
        has_error = False
    
        for i in range(end - (start + 1)):
            cell = cells[i + start + 1]
            if not has_header2(cell):
                images = get_images_from_cell(i + start + 1, cell)
                if images:
                    last_images = images
                continue
            if last_images is not None and len(last_images) > 0:
                screenshot = openpyxl.drawing.image.Image(last_images[0])
                screenshot.height = itemheight
                screenshot.width = int(itemheight / 1080 * 1920)
                shutil.copy(last_images[0], os.path.join(result_dir, 'screenshots', test_id, '{0:05d}.png'.format(itemindex - 1)))
            # 成功したか？
            output_types = []
            outputs = []
            for next_cell in cells[i + start + 1 + 1:]:
                if has_header2(next_cell):
                    break
                if 'outputs' not in next_cell:
                    continue
                #assert all([o['output_type'] != 'error' for o in next_cell['outputs']]), next_cell['outputs']
                output_types += [o['output_type'] for o in next_cell['outputs']]
                outputs += list(next_cell['outputs'])
            output_types = set(output_types)
            if 'error' in output_types or len(output_types) == 0:
                has_error = True
            line = cell['source'].split('\n')[0]
            m = re.match(r'##\s+(.+)', line)
            row = startrow + itemindex
            sheet[f'A{row}'] = str(itemindex)
            sheet[f'B{row}'] = m.group(1)
            sheet[f'C{row}'] = '\n'.join(cell['source'].split('\n')[1:]).strip()
            sheet[f'D{row}'] = '■' if 'error' not in output_types and len(output_types) > 0 else '□'
            sheet[f'E{row}'] = '' if 'error' not in output_types else '\n'.join([o['evalue'] if 'evalue' in o else o['ename'] for o in outputs if o['output_type'] == 'error'])
            sheet[f'F{row}'] = 'Playwright'
            sheet[f'G{row}'] = datetime.now().strftime('%Y-%m-%d')
            sheet[f'H{row}'] = ''
            for cell in sheet[f'A{row}:H{row}'][0]:
                cell.alignment = Alignment(wrap_text=True, vertical='top')
            sheet[f'D{row}'].alignment = Alignment(wrap_text=True, vertical='top', horizontal='center')
            
            sheet.row_dimensions[row].height = itemheight
    
            itemindex += 1
            last_images = None
            
        if last_images is not None and len(last_images) > 0:
            screenshot = openpyxl.drawing.image.Image(last_images[0])
            screenshot.height = itemheight
            screenshot.width = int(itemheight / 1080 * 1920)
            shutil.copy(last_images[0], os.path.join(result_dir, 'screenshots', test_id, '{0:05d}.png'.format(itemindex - 1)))            
            
        summaryrow = index + 1
        summary_sheet[f'A{summaryrow}'] = test_id
        summary_sheet[f'B{summaryrow}'] = test_id
        summary_sheet[f'C{summaryrow}'] = attrs['サブシステム名']
        summary_sheet[f'D{summaryrow}'] = attrs['ページ/アドオン']
        summary_sheet[f'E{summaryrow}'] = attrs['機能分類']
        summary_sheet[f'F{summaryrow}'] = attrs['シナリオ名']
        summary_sheet[f'G{summaryrow}'] = title
        summary_sheet[f'H{summaryrow}'] = f'参照: {test_id}'
        summary_sheet[f'H{summaryrow}'].hyperlink = f'#{test_id}!A1'
        summary_sheet[f'I{summaryrow}'] = '成功' if not has_error else '失敗'
        summary_sheet[f'J{summaryrow}'] = f'GRDM-{ticket_number}'
        summary_sheet[f'J{summaryrow}'].hyperlink = f'https://redmine.devops.rcos.nii.ac.jp/issues/{ticket_number}'
        summary_sheet[f'K{summaryrow}'] = author
        summary_sheet[f'L{summaryrow}'] = datetime.now().strftime('%Y-%m-%d')
        for cell in summary_sheet[f'A{summaryrow}:O{summaryrow}'][0]:
            cell.alignment = Alignment(wrap_text=True, vertical='top')

today = datetime.now().strftime('%Y-%m-%d')

filename_prefix = os.path.join(result_dir, 'test-summary')
print(f'Saved: {filename_prefix}-{today}.xlsx')
wb.save(os.path.join(f'{filename_prefix}-{today}.xlsx'))

In [13]:
!cd {result_dir}; zip test-summary-{today}.zip *.xlsx -r screenshots

## 統計情報の収集

In [14]:
import pandas as pd
import json

har_dfs = []
for dirname in os.listdir(result_dir):
    dirpath = os.path.join(result_dir, dirname)
    if not os.path.isdir(dirpath):
        continue
    har_path = os.path.join(dirpath, 'har.zip')
    items = []
    if not os.path.exists(har_path):
        continue
    if os.path.exists(os.path.join(work_dir, 'har')):
        !rm -fr {work_dir}/har
    !mkdir -p {work_dir}/har
    !unzip -d {work_dir}/har "{har_path}"
    har_json_path = os.path.join(work_dir, 'har', 'har.har')
    assert os.path.exists(har_json_path)
    with open(har_json_path, 'r') as f:
        har_json = json.load(f)
        if 'log' in har_json and 'entries' in har_json['log']:
            for e in har_json['log']['entries']:
                items.append({
                    'time': e['time'],
                    'method': e['request']['method'],
                    'url': e['request']['url'],
                    'status': e['response']['status'],
                })
    har_df = pd.DataFrame(items)
    har_df['test'] = dirname
    har_dfs.append(har_df)
har_df = pd.concat(har_dfs)

last_5xx_urls = ''
if len(har_df[har_df['status'] >= 500]) > 0:
    last_5xx_urls = 'time, url, status\n' + '\n'.join([f'{t}, {url}, {status}' for i, (t, url, status) in enumerate(har_df[har_df['status'] >= 500][['time', 'url', 'status']].values)])
else:
    last_5xx_urls = ''
last_5xx_urls

''

In [15]:
har_df

In [16]:
har_df.sort_values('time')

In [17]:
har_df[~har_df['status'].isin([200, 201, 404, 301, 302, 304, 308, 410])].sort_values('time')

In [18]:
har_df.sort_values('time').to_csv(os.path.join(result_dir, 'response.csv'))

In [19]:
!rm -fr {work_dir}